In [14]:
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

In [ ]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [ ]:
# Reused from GammaMixture.ipynb / [MDN] theta to gamma coeffs.ipynb -- keep in
# sync if those notebooks' loader changes.
def load_all_trial_bkt_data(mat_path, ps_indices=None):
    param_sets = {}
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        indices = ps_indices if ps_indices is not None else range(atbd.shape[1])
        for i in indices:
            ps_grp = f[atbd[0, i]]
            p0 = float(ps_grp['p0'][()].flat[0])
            td = ps_grp['trial_data']
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                trial_data.append(arr.flatten() if arr.dtype == np.float64 else np.array([], dtype=np.float64))
            param_sets[i] = {'p0': p0, 'trial_data': trial_data}
    return param_sets, weights


def recover_theta(w):
    disc = 1*w[0] + 2.1*(1 - w[0])           # [1, 2.1]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])  # [0.01e-3, 0.3e-3]
    lr   = 0.3*w[2] + 0.5*(1 - w[2])         # [0.3, 0.5]
    return disc, lr, b


def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial from its raw particle
    arrival times (order statistic), or None if undefined. Copied from
    [per p] distrib of pth percentile bkt.ipynb."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None

In [ ]:
# Same connectivity filter as [MDN] theta to gamma coeffs.ipynb.
P0_THRESHOLD = 0.9
Np = 1000

with h5py.File(mat_path, 'r') as f:
    n_param_sets = f['all_trial_bkt_data'].shape[1]

all_param_sets, weights = load_all_trial_bkt_data(mat_path, range(n_param_sets))
USABLE_PS = sorted(i for i, ps in all_param_sets.items() if ps['p0'] < P0_THRESHOLD)
print(f"{len(USABLE_PS)} / {n_param_sets} param sets usable (p0 < {P0_THRESHOLD})")

In [ ]:
# Is a 2-component Gaussian mixture (in log space) actually sufficient for the
# distribution of pth-percentile BKT across trials, at fixed (theta,p)? Check
# via BIC across K=1,2,3 for a spread of param sets (by connectivity) and
# percentiles, rather than assuming.
CHECK_P_VALS = [0.01, 0.1, 0.5, 0.9]
ps_by_p0 = sorted(USABLE_PS, key=lambda i: all_param_sets[i]['p0'])
CHECK_PS = [ps_by_p0[0], ps_by_p0[len(ps_by_p0) // 2], ps_by_p0[-1]]  # well-connected / mid / marginal

fig, axes = plt.subplots(len(CHECK_PS), len(CHECK_P_VALS), figsize=(4 * len(CHECK_P_VALS), 3.2 * len(CHECK_PS)))
for row, ps_idx in enumerate(CHECK_PS):
    ps = all_param_sets[ps_idx]
    for col, p in enumerate(CHECK_P_VALS):
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        ax = axes[row, col]
        if len(vals) < 10:
            ax.set_title(f'ps{ps_idx} p={p} (n={len(vals)}, too few)')
            continue
        logvals = np.log(vals).reshape(-1, 1)
        bics = {k: GaussianMixture(n_components=k, random_state=0, n_init=3).fit(logvals).bic(logvals)
                for k in [1, 2, 3]}
        best_k = min(bics, key=bics.get)
        ax.hist(logvals, bins=40, density=True, alpha=0.5)
        xg = np.linspace(logvals.min(), logvals.max(), 300).reshape(-1, 1)
        for k in [1, 2, 3]:
            gm = GaussianMixture(n_components=k, random_state=0, n_init=3).fit(logvals)
            ax.plot(xg, np.exp(gm.score_samples(xg)), label=f'K={k} BIC={bics[k]:.0f}',
                     lw=1.5 if k == best_k else 1, ls='-' if k == best_k else '--')
        ax.set_title(f'ps{ps_idx} (p0={ps["p0"]:.2f}) p={p}\nn={len(vals)}, best K={best_k}')
        ax.legend(fontsize=6)
plt.tight_layout()
plt.savefig(transport_dir / 'percentile_gmm_K_check.png', dpi=110)
plt.show()

# Finding (see conclusion cell): 2 components is a reasonable default -- matches
# GMM.ipynb's precedent, and clearly beats K=1 for well-connected/high-sample
# param sets (skewed, long-tailed shape). But it's not obviously "correct":
# K=3 keeps improving BIC further for high-n groups (mild residual tail
# structure), while low-sample/marginal-connectivity groups actually PREFER
# K=1 (2 components risks fitting noise there). We proceed with K=2 as
# requested, but this is the evidence behind that choice, not an assumption.
K = 2

In [ ]:
# Precompute stage: fit a K-component GMM (log space) to the distribution,
# across trials, of the pth-percentile BKT for every (param set, p) pair.
# Cheap (~7s for ~1500 groups) since this is a plain sklearn GMM fit on a
# 1-D order statistic, unlike the 2-component GAMMA MIXTURE MLE fit used in
# [MDN] theta to gamma coeffs.ipynb (~0.5s/fit, needed multiprocessing).
# p is essentially free to sample densely (no new simulation needed, unlike
# theta), so we use a much finer p-grid than the 8 values used elsewhere.
p_vals = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3,
                    0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
MIN_SAMPLES = 20

records = []
for ps_idx in USABLE_PS:
    ps = all_param_sets[ps_idx]
    disc, lr, b = recover_theta(weights[ps_idx])
    for p in p_vals:
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        if len(vals) < MIN_SAMPLES:
            continue
        logvals = np.log(vals).reshape(-1, 1)
        gm = GaussianMixture(n_components=K, random_state=0, n_init=3, reg_covar=1e-6).fit(logvals)
        order = np.argsort(gm.means_.flatten())  # sort by mean ascending, avoids label-switching
        w = gm.weights_[order]
        mu = gm.means_.flatten()[order]
        sigma = np.sqrt(gm.covariances_.flatten()[order])
        records.append({
            'paramset_idx': ps_idx, 'discspacing': disc, 'length_ratio': lr, 'b': b, 'p': p,
            'n_samples': len(vals),
            'w1': w[0], 'mu1': mu[0], 'sigma1': sigma[0], 'mu2': mu[1], 'sigma2': sigma[1],
        })

gmm_df = pd.DataFrame(records)
print(f"Fit {len(gmm_df)} (theta,p) groups across {gmm_df['paramset_idx'].nunique()} param sets")

In [ ]:
# Build the regression dataset: input (discspacing, length_ratio, b, logit(p)),
# target (w1, mu1, sigma1, mu2, sigma2), both z-scored (same convention as
# GMM.ipynb). logit(p) rather than raw p since p is a probability.
FEATURE_COLS = ['discspacing', 'length_ratio', 'b']
TARGET_COLS = ['w1', 'mu1', 'sigma1', 'mu2', 'sigma2']


def logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


X_all = np.column_stack([gmm_df[FEATURE_COLS].values, logit(gmm_df['p'].values)])
Y_all = gmm_df[TARGET_COLS].values

X_mean, X_std = X_all.mean(axis=0), X_all.std(axis=0)
Y_mean, Y_std = Y_all.mean(axis=0), Y_all.std(axis=0)


def normalize_X(X):
    return (X - X_mean) / X_std


def normalize_Y(Y):
    return (Y - Y_mean) / Y_std


def denormalize_Y(Y):
    return Y * Y_std + Y_mean

In [ ]:
# Regression network -- same architecture/training recipe as GMM.ipynb
# (Dense 4->20->20->20->20->5, L2=9e-5, weighted MSE against the precomputed
# GMM coefficients, Adam with a stepped LR schedule), just with the input
# widened from 3 to 4 (theta + p) and the output narrowed from 7 to 5 (no
# p0/diagnostic -- out of scope here, connectivity is already handled by the
# P0_THRESHOLD filter above).
def build_model():
    l2_reg = 9e-5
    return keras.Sequential([
        keras.layers.Input(shape=(4,)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(20, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(5),
    ])


_loss_weights = tf.constant([2.0, 3.0, 2.0, 3.0, 2.0], dtype=tf.float32)  # w1,mu1,sigma1,mu2,sigma2
_loss_weights = _loss_weights / tf.reduce_sum(_loss_weights)


def weighted_mse(y_true, y_pred):
    return tf.reduce_mean(tf.reduce_sum(_loss_weights * tf.square(y_true - y_pred), axis=1))


def lr_by_epoch(epoch):
    if epoch < 200:
        return 1e-3
    elif epoch < 300:
        return 5e-4
    else:
        return 1e-4


def train_model(X_train, Y_train, X_val, Y_val, epochs=400, verbose=0):
    model = build_model()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=weighted_mse)
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs,
                         batch_size=32, callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch, verbose=0)],
                         verbose=verbose)
    return model, history

In [ ]:
# Train the main model on a random 80/20 split (matches GMM.ipynb's convention).
X_norm, Y_norm = normalize_X(X_all), normalize_Y(Y_all)
X_train, X_val, Y_train, Y_val = train_test_split(X_norm, Y_norm, test_size=0.2, random_state=42)

model, history = train_model(X_train, Y_train, X_val, Y_val, epochs=400, verbose=0)
print(f"final train loss: {history.history['loss'][-1]:.4f}  val loss: {history.history['val_loss'][-1]:.4f}")
model.save(transport_dir / 'percentile_gmm_model.keras')

In [15]:
# For every usable param set, plot the empirical pth-percentile-BKT
# distribution (solid, color-coded by p, same style/p_vals/colormap as
# [per p] distrib of pth percentile bkt.ipynb) with the NN-predicted
# 2-Gaussian-mixture PDF for that (theta,p) overlaid in the same color but
# dashed -- one page per param set in a multi-page PDF. Uses the cell-8
# `model` (trained on the full dataset) as a full-dataset visual QC pass;
# cell-9's leave-one-out CV already covers held-out generalization.
plot_p_vals = [0.001, 0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]  # same as [per p]...ipynb
plot_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(plot_p_vals)))
LOG10 = np.log(10)  # mu/sigma were fit in natural-log space; convert to log10 to match [per p]'s x-axis

# batch all (param set, p) predictions in one call rather than ~700 individual ones
plot_rows = [(ps_idx, p) for ps_idx in USABLE_PS for p in plot_p_vals]
X_plot = np.array([[*recover_theta(weights[ps_idx]), logit(p)] for ps_idx, p in plot_rows])
Y_plot_pred = denormalize_Y(model.predict(normalize_X(X_plot), verbose=0))
pred_lookup = {key: Y_plot_pred[i] for i, key in enumerate(plot_rows)}

pdf_path = transport_dir / '[per p] distrib of pth percentile bkt, empirical vs NN.pdf'
with PdfPages(pdf_path) as pdf:
    for ps_idx in USABLE_PS:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])

        fig, ax = plt.subplots(figsize=(9, 6))
        for p, color in zip(plot_p_vals, plot_colors):
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            if len(vals) < 2:
                continue

            log_vals = np.log10(vals)
            counts, edges = np.histogram(log_vals, bins=60, density=True)
            centres = (edges[:-1] + edges[1:]) / 2
            ax.plot(centres, counts, color=color, linestyle='-', label=f'p={p:.3f} (n={len(vals)})')

            w1, mu1, sigma1, mu2, sigma2 = pred_lookup[(ps_idx, p)]
            mu1_10, sigma1_10 = mu1 / LOG10, sigma1 / LOG10
            mu2_10, sigma2_10 = mu2 / LOG10, sigma2 / LOG10
            xg = np.linspace(centres.min() - 0.3, centres.max() + 0.3, 300)
            pred_pdf = (w1 * np.exp(-0.5 * ((xg - mu1_10) / sigma1_10) ** 2) / (sigma1_10 * np.sqrt(2 * np.pi)) +
                        (1 - w1) * np.exp(-0.5 * ((xg - mu2_10) / sigma2_10) ** 2) / (sigma2_10 * np.sqrt(2 * np.pi)))
            ax.plot(xg, pred_pdf, color=color, linestyle='--')

        ax.set_xlabel('breakthrough time (s)')
        ax.set_ylabel('probability density')
        ticks = ax.get_xticks()
        ax.set_xticks(ticks)
        ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
        ax.set_title(f'Param set {ps_idx}  disc={disc:.2f}, lr={lr:.2f}, b={b:.2e}, p0={ps["p0"]:.2f}\n'
                     f'solid = empirical, dashed = NN-predicted GMM')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}  ({len(USABLE_PS)} pages)')

Saved /Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP/outputs/[per p] distrib of pth percentile bkt, empirical vs NN.pdf  (87 pages)


In [ ]:
# --- Conclusion ---
#
# On whether 2 Gaussians suffices (cell 4): it's a reasonable default, matching
# GMM.ipynb's precedent, and clearly better than 1 for well-connected/
# high-sample param sets (right-skewed, long-tailed shape) -- but BIC keeps
# improving somewhat at K=3 for those same high-n groups (mild residual tail
# structure not captured by 2 components), while low-sample/marginal-
# connectivity groups actually PREFER K=1 by BIC (2 components risks fitting
# noise there). So: not obviously "wrong" to use 2, but also not clearly
# sufficient everywhere -- if this becomes a bigger effort, making K tunable
# (as in [MDN] theta to gamma coeffs.ipynb) rather than fixed would be the
# natural next step.
#
# On whether theta helps beyond p alone (cell 9): see the printed model-vs-
# baseline comparison above. Unlike the 5-D joint-coefficient MDN (which
# needed ~87 distinct theta points before beating a theta-blind baseline),
# this network only has to learn a 5-D *regression* target (not a full joint
# density via NLL), which is a substantially easier learning problem --
# worth checking whether that shows up as an easier win here.